# 🛰️ Satellite Vegetation Monitor (v2)
### Hands-On Activity · Lesson 6 — Geospatial AI and Satellite Data
**AI for Sustainable Development · ESDU @ AUB · karianet.org**

---

## How this activity works

This activity has **two parts**, in two different places:

| Part | Where | What you do |
|------|-------|-------------|
| **A — Concepts & Setup** | This notebook | Read the explanations, register for Earth Engine |
| **B — Run the Analysis** | The **Earth Engine Code Editor** (a website, not this notebook) | Copy-paste one script, click Run, read your results |
| **C — Reflect & Submit** | Back in this notebook | Enter your results, answer reflections, submit |

**Why two places?** Earth Engine's full power runs in its own free
browser-based tool called the **Code Editor** — built by Google
specifically so satellite analysis doesn't require installing anything
or writing complex setup code. We use it directly instead of fighting
with Python installations. You will copy and paste ONE script — no
programming experience required, every line is explained.

Let's begin.


## Step 0 — Enter your details

In [ ]:
your_name  = "Your Full Name"   # ← replace
your_email = "your@email.com"   # ← replace
print(f"Name : {your_name}")
print(f"Email: {your_email}")


## Part A.1 — What Are We Actually Doing in This Activity?

We are going to measure **plant health from space** across an entire
agricultural region, using free satellite data, and watch how that
health changes across the four seasons of one year.

This is the same basic workflow used by real agronomists, NGOs, and
government agencies to:
- Detect drought stress before it's visible from the ground
- Predict crop yield weeks before harvest
- Decide where to send water, fertilizer, or aid
- Track land degradation or desertification over years

By the end of this activity, you will have produced the same kind of
output a professional remote-sensing analyst would generate — just on
a smaller, more manageable scale.


## Part A.2 — Core Concept: NDVI (Read This Before You Touch Any Code)

**NDVI** stands for **Normalized Difference Vegetation Index**.
It is the single most widely used number in all of satellite agriculture.

### The idea in plain language
Healthy, growing plants do something curious: they strongly **reflect**
a type of light called Near-Infrared (NIR) — light just outside what
human eyes can see — while **absorbing** most red light for photosynthesis.

Stressed, dying, or absent plants do the opposite: they reflect much
less Near-Infrared and more red light bounces back unused.

Satellites carry sensors that can measure both Near-Infrared and Red
light reflected from the ground. NDVI is simply the formula that turns
those two measurements into a single number:

```
NDVI = (Near-Infrared − Red) / (Near-Infrared + Red)
```

### How to read an NDVI number

| NDVI value | What it usually means |
|------------|------------------------|
| Below 0 | Water, cloud, or shadow — not land vegetation |
| 0.0 – 0.2 | Bare soil, rock, or very sparse vegetation |
| 0.2 – 0.5 | Moderate vegetation — could be early-growth crops or stressed plants |
| 0.5 – 1.0 | Dense, healthy, actively growing vegetation |

You do not need to calculate this formula yourself — the script in
Part B does it automatically. You just need to understand what the
number means when you see it.

### A second number: NDWI

We'll also calculate **NDWI** (Normalized Difference Water Index),
which works the same way but measures water content rather than
greenness. Higher NDWI = more water in the plant tissue — useful for
spotting drought stress even before NDVI drops.


## Part A.3 — Register for Google Earth Engine (One-Time, ~5 Minutes)

If you've already registered, skip to Part A.4.

1. Go to **https://earthengine.google.com/**
2. Click **"Get Started"**
3. Sign in with any Google account
4. Click **"Register a Noncommercial or Commercial Cloud Project"**
5. Select **"Unpaid usage"** → choose **"Academia & Research"**
6. Give your project any name (e.g. "karianet-ai-course")
7. Submit — approval for academic/noncommercial use is usually instant

Once registered, you can access the Code Editor anytime at:
**https://code.earthengine.google.com**


## Part A.4 — Get the Script

Run the cell below. It will display the full Earth Engine script
**inside this notebook** so you can copy it directly — you don't need
to open a separate file.


In [ ]:
gee_script = '''
/* ════════════════════════════════════════════════════════════════════════
   SATELLITE VEGETATION MONITOR — Google Earth Engine Script
   AI for Sustainable Development · Lesson 6 · ESDU @ AUB · karianet.org
   ════════════════════════════════════════════════════════════════════════

   HOW TO USE THIS SCRIPT
   ───────────────────────
   1. Go to https://code.earthengine.google.com and sign in
   2. Delete anything in the empty script panel (center of the screen)
   3. Copy this ENTIRE script and paste it into that empty panel
   4. Click the blue "Run" button above the panel (or press Ctrl+Enter)
   5. Wait 10-30 seconds — satellite data is being processed on Google's servers
   6. Look at THREE places for your results:
        • MAP (bottom panel)     -> the visual map layers
        • CONSOLE (top right)    -> printed text and charts
        • Console "Apps" charts  -> the NDVI time-series chart

   You do NOT need to know JavaScript to complete this activity.
   Every section below is explained in plain language before the code.
   You only need to EDIT the values marked with the pencil symbol --
   everything else can be copy-pasted exactly as written.
   ════════════════════════════════════════════════════════════════════════ */


/* ──────────────────────────────────────────────────────────────────────
   SECTION 1 -- DEFINE YOUR STUDY AREA
   ────────────────────────────────────────────────────────────────────── */

// EDIT THESE FOUR NUMBERS if you want a different study area
var studyArea = ee.Geometry.Rectangle([35.85, 33.55, 36.30, 33.90]);

// EDIT THIS YEAR if you want to study a different year (2017-2026 available)
var YEAR = 2023;

Map.centerObject(studyArea, 9);
Map.addLayer(studyArea, {color: "red"}, "My Study Area (outline)");

print("SECTION 1 complete -- study area defined.");
print("Look at the MAP panel below -- you should see a red rectangle.");


/* ──────────────────────────────────────────────────────────────────────
   SECTION 2 -- LOAD SENTINEL-2 SATELLITE IMAGERY
   ────────────────────────────────────────────────────────────────────── */

function getCloudFreeImage(startDate, endDate) {
  var collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(startDate, endDate)
    .filterBounds(studyArea)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .map(function(img) { return img.divide(10000); });

  var composite = collection.median().clip(studyArea);
  return {image: composite, count: collection.size()};
}

print("SECTION 2 complete -- image-loading function defined.");


/* ──────────────────────────────────────────────────────────────────────
   SECTION 3 -- CALCULATE NDVI AND NDWI
   ────────────────────────────────────────────────────────────────────── */

function addVegetationIndices(image) {
  var ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI");
  var ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI");
  return image.addBands(ndvi).addBands(ndwi);
}

print("SECTION 3 complete -- NDVI/NDWI calculation function defined.");


/* ──────────────────────────────────────────────────────────────────────
   SECTION 4 -- LOAD AND VISUALIZE FOUR SEASONS
   ────────────────────────────────────────────────────────────────────── */

var seasons = [
  {name: "Winter", start: YEAR + "-01-01", end: YEAR + "-03-31"},
  {name: "Spring", start: YEAR + "-04-01", end: YEAR + "-06-30"},
  {name: "Summer", start: YEAR + "-07-01", end: YEAR + "-09-30"},
  {name: "Autumn", start: YEAR + "-10-01", end: YEAR + "-12-31"}
];

var ndviVis = {
  min: -0.2,
  max: 0.8,
  palette: ["red", "orange", "yellow", "lightgreen", "darkgreen"]
};

var seasonalNDVI = {};

seasons.forEach(function(season) {
  var result = getCloudFreeImage(season.start, season.end);
  var withIndices = addVegetationIndices(result.image);
  var ndviLayer = withIndices.select("NDVI");
  seasonalNDVI[season.name] = ndviLayer;
  Map.addLayer(ndviLayer, ndviVis, "NDVI - " + season.name, false);
  print("Loaded " + season.name + ": " + season.start + " to " + season.end);
});

print("SECTION 4 complete -- 4 seasonal NDVI layers added to the map.");
print("Go to the MAP panel, find the layer list (top-right corner of map),");
print("and tick each NDVI layer checkbox one at a time to compare them.");
print("Green areas = healthy vegetation. Red/orange areas = stressed or bare.");


/* ──────────────────────────────────────────────────────────────────────
   SECTION 5 -- CHART: NDVI ACROSS THE YEAR
   ────────────────────────────────────────────────────────────────────── */

var chartData = seasons.map(function(season) {
  var ndviImage = seasonalNDVI[season.name];
  var meanDict = ndviImage.reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: studyArea,
    scale: 100,
    maxPixels: 1e9
  });
  return ee.Feature(null, {season: season.name, NDVI: meanDict.get("NDVI")});
});

var chartFeatureCollection = ee.FeatureCollection(chartData);

var ndviChart = ui.Chart.feature.byFeature(chartFeatureCollection, "season", ["NDVI"])
  .setChartType("LineChart")
  .setOptions({
    title: "Mean NDVI by Season -- " + YEAR,
    hAxis: {title: "Season"},
    vAxis: {title: "Mean NDVI", viewWindow: {min: -0.1, max: 0.9}},
    colors: ["2e7d32"],
    lineWidth: 3,
    pointSize: 7
  });

print(ndviChart);
print("SECTION 5 complete -- chart printed above. Scroll up in the Console to see it.");


/* ──────────────────────────────────────────────────────────────────────
   SECTION 6 -- PRINT THE EXACT NUMBERS FOR YOUR SUBMISSION
   ────────────────────────────────────────────────────────────────────── */

print("===============================================");
print("YOUR RESULTS -- copy these numbers into your notebook:");
print("===============================================");

seasons.forEach(function(season) {
  var ndviImage = seasonalNDVI[season.name];
  var meanDict = ndviImage.reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: studyArea,
    scale: 100,
    maxPixels: 1e9
  });
  meanDict.get("NDVI").evaluate(function(value) {
    print(season.name + " mean NDVI: " + value.toFixed(3));
  });
});

print("ALL SECTIONS COMPLETE.");
print("Scroll through the Console above to find your 4 NDVI values.");
print("Go back to your Jupyter notebook and enter them into the results cell.");
'''

print("✅ Script loaded into the variable 'gee_script'.")
print("Run the next cell to display it in a copy-friendly box.")


In [ ]:
from IPython.display import display, HTML
import html as _html

escaped = _html.escape(gee_script)
display(HTML(f'''
<div style="background:#1e1e1e;color:#d4d4d4;padding:16px;border-radius:8px;
            font-family:monospace;font-size:12px;max-height:500px;overflow-y:scroll;
            white-space:pre-wrap;border:2px solid #2e7d32;">
{escaped}
</div>
'''))
print("\n👆 Select all the text in the box above (click inside, Ctrl+A, Ctrl+C)")
print("   and paste it into a new script at code.earthengine.google.com")


## Part B — Run the Script in the Earth Engine Code Editor

Now switch to your browser and follow these steps:

1. **Open** [code.earthengine.google.com](https://code.earthengine.google.com) in a new tab
2. If this is your first visit, you may be asked to create a **home folder** —
   pick any name (e.g. your Google username) and confirm
3. In the **center panel**, you'll see an empty script editor — click inside it
4. **Paste** the script you copied above (Ctrl+V)
5. Click the blue **Run** button above the editor (or press **Ctrl+Enter**)
6. Wait — Earth Engine is processing real satellite data on Google's servers,
   this can take 10–30 seconds
7. **Three things will happen:**
   - The **Console** (top-right panel) fills with text — read through it, each
     line explains what just happened
   - A red rectangle appears on the **Map** (bottom panel) — that's your study area
   - A **chart** appears in the Console — this is your NDVI time-series

### Exploring the map layers

In the top-right corner of the **map** itself (not the console), there's a small
**Layers** icon. Click it to see a list of layers — tick the checkboxes for
"NDVI - Winter", "NDVI - Spring", etc. one at a time to compare seasons visually.
Green = healthy vegetation. Red/orange = stressed or bare ground.

### Finding your numbers

Scroll to the **bottom of the Console** — Section 6 of the script prints the
exact mean NDVI value for each of the 4 seasons as plain text, like:

```
Winter mean NDVI: 0.342
Spring mean NDVI: 0.518
Summer mean NDVI: 0.221
Autumn mean NDVI: 0.295
```

**Write these four numbers down** — you'll enter them in Part C below.

> **Troubleshooting:** If you see a red error message in the console instead of
> results, check that you copied the *entire* script (Section 1 through the
> final comment block) and that nothing was cut off. Re-copy and paste again
> if needed — this is the most common issue and is always fixed by re-pasting.


## Part C.1 — Enter Your Results

Now back in this notebook. Type in the four NDVI values you found in the
Earth Engine Console.


In [ ]:
# ── YOUR RESULTS FROM THE GEE CONSOLE ──────────────────────────────────
# Replace the 0.000 values with your actual results from Section 6 of the console output

my_ndvi_results = {
    "Winter": 0.000,   # ← replace with your value
    "Spring": 0.000,   # ← replace with your value
    "Summer": 0.000,   # ← replace with your value
    "Autumn": 0.000,   # ← replace with your value
}

print("Your seasonal NDVI results:")
for season, value in my_ndvi_results.items():
    interp = ("Dense vegetation" if value > 0.5 else
              "Moderate vegetation" if value > 0.3 else
              "Sparse/stressed" if value > 0.1 else
              "Bare soil / water / not yet entered")
    print(f"  {season:<10} {value:.3f}   →  {interp}")


## Part C.2 — Visualize Your Results Here Too

Run this cell to redraw your Earth Engine chart inside the notebook,
using the numbers you just typed in. This gives you a second copy of
your chart to include in your submission.


In [ ]:
import matplotlib.pyplot as plt

seasons_order = ["Winter", "Spring", "Summer", "Autumn"]
values = [my_ndvi_results[s] for s in seasons_order]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seasons_order, values, "o-", color="#2e7d32", linewidth=2.5,
        markersize=9, markerfacecolor="#1b5e20")
for x, y in zip(seasons_order, values):
    ax.annotate(f"{y:.3f}", (x, y), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=10)

ax.axhline(0.2, color="orange", linestyle="--", alpha=0.6, label="Vegetation threshold (0.2)")
ax.axhline(0.5, color="green", linestyle="--", alpha=0.4, label="Dense vegetation (0.5)")
ax.set_title("Mean NDVI by Season (from your GEE results)", fontsize=13, fontweight="bold")
ax.set_ylabel("Mean NDVI")
ax.set_ylim(-0.1, 0.9)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("my_ndvi_chart.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Chart saved as my_ndvi_chart.png")


## Part C.3 — Reflection Questions

In [ ]:
# Reflection 1
# Looking at your NDVI chart, describe what you observe.
# Which season shows the most vegetation stress, and what might cause this
# in the Bekaa Valley or a similar MENA agricultural context?

reflection_1 = """
Write your answer here. Reference your actual NDVI numbers.
"""
print(reflection_1.strip() if len(reflection_1.strip()) > 20 else "❌ Write at least 2 sentences referencing your numbers.")


In [ ]:
# Reflection 2
# How could a farmer or agricultural NGO use seasonal NDVI monitoring
# to make better decisions? Give one concrete example with a specific
# action triggered by a specific NDVI reading.

reflection_2 = """
Write your answer here.
"""
print(reflection_2.strip() if len(reflection_2.strip()) > 20 else "❌ Write at least 2 sentences.")


## Part C.4 — Upload Your Screenshots

Before submitting, upload **two screenshots** from the Earth Engine Code Editor:
1. A screenshot of the **map** with at least one NDVI layer visible
2. A screenshot of the **chart** from the Console

Click the folder icon on the left sidebar of Colab, then drag and drop both
images into the file area. Then enter their filenames below.


In [ ]:
# ── SCREENSHOT FILENAMES ───────────────────────────────────────────────
# After uploading via the Colab file panel (folder icon, left sidebar),
# type the exact filenames here

map_screenshot_filename   = "map_screenshot.png"     # ← replace with your filename
chart_screenshot_filename = "chart_screenshot.png"   # ← replace with your filename

import os
for label, fname in [("Map screenshot", map_screenshot_filename),
                     ("Chart screenshot", chart_screenshot_filename)]:
    status = "✅ found" if os.path.exists(fname) else "❌ not found — check filename or upload it"
    print(f"{label}: {fname}  →  {status}")


## Part C.5 — Submit Your Activity

In [ ]:
import ipywidgets as widgets, json, hashlib, re, base64, os
from datetime import datetime
from IPython.display import display

_btn = widgets.Button(description="Submit Activity", button_style="success",
                      layout=widgets.Layout(width="220px", height="40px"))
_out = widgets.Output()

def _submit(b):
    _out.clear_output()
    with _out:
        errs = []
        if len(your_name.strip()) < 3 or your_name.strip() == "Your Full Name":
            errs.append("❌ Enter your full name in Step 0.")
        if "@" not in your_email or your_email.strip() == "your@email.com":
            errs.append("❌ Enter a valid email in Step 0.")
        if all(v == 0.000 for v in my_ndvi_results.values()):
            errs.append("❌ Enter your actual NDVI results from the GEE Console (Part C.1).")
        if not os.path.exists("my_ndvi_chart.png"):
            errs.append("❌ Run Part C.2 to generate your chart.")
        if not os.path.exists(map_screenshot_filename):
            errs.append(f"❌ Map screenshot not found: {map_screenshot_filename}")
        if not os.path.exists(chart_screenshot_filename):
            errs.append(f"❌ Chart screenshot not found: {chart_screenshot_filename}")
        if len(reflection_1.strip()) < 30:
            errs.append("❌ Reflection 1 is too short.")
        if len(reflection_2.strip()) < 30:
            errs.append("❌ Reflection 2 is too short.")

        if errs:
            for e in errs: print(e)
            print("\nFix the items above and click Submit Activity again.")
            return

        images_b64 = {}
        for label, fname in [("notebook_chart", "my_ndvi_chart.png"),
                             ("map_screenshot", map_screenshot_filename),
                             ("chart_screenshot", chart_screenshot_filename)]:
            with open(fname, "rb") as f:
                images_b64[label] = base64.b64encode(f.read()).decode()

        payload = {
            "activity": "Satellite Vegetation Monitor (v2 - GEE Code Editor)",
            "lesson": "Lesson 6 — Geospatial AI and Satellite Data",
            "name": your_name.strip(),
            "email": your_email.strip(),
            "ndvi_results": my_ndvi_results,
            "images_base64": images_b64,
            "reflection_1": reflection_1.strip(),
            "reflection_2": reflection_2.strip(),
            "timestamp": datetime.utcnow().isoformat() + "Z",
        }
        raw = json.dumps({k: v for k, v in payload.items() if k != "images_base64"},
                         sort_keys=True, ensure_ascii=False, default=str)
        payload["checksum"] = hashlib.sha256(raw.encode()).hexdigest()

        slug  = re.sub(r"[^a-zA-Z0-9]+", "_", your_name.strip().lower()).strip("_") or "student"
        fname = f"submission_{slug}.json"
        with open(fname, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False, default=str)

        print(f"🎉 Submission ready: {fname}")
        print(f"   NDVI results: {my_ndvi_results}")
        print(f"   Checksum: {payload['checksum'][:16]}...")
        try:
            from google.colab import files
            files.download(fname)
            print("\n⬇️  Download started — check your Downloads folder.")
            print("📌 Upload this file to the assignment box on Karianet.")
        except ImportError:
            print("\nℹ️  Not in Colab — find file in the Files panel.")

_btn.on_click(_submit)
display(_btn, _out)


---
## Submission Checklist
- [ ] Name and email entered (Step 0)
- [ ] Registered for Google Earth Engine (Part A.3)
- [ ] Script copied and run successfully in the Code Editor (Part B)
- [ ] Found all 4 seasonal NDVI values in the Console
- [ ] Entered the 4 NDVI values in this notebook (Part C.1)
- [ ] Chart regenerated in this notebook (Part C.2)
- [ ] Both reflections written, referencing your actual numbers
- [ ] Map and chart screenshots uploaded and filenames entered (Part C.4)
- [ ] Submit button clicked and `submission_[yourname].json` downloaded
- [ ] File uploaded to the assignment box on Karianet

---
*Satellite Vegetation Monitor v2 · AI for Sustainable Development · Lesson 6 · ESDU @ AUB*
